# Stage 13 — Productization

## 1. Goal

Train and save a two-feature model, serve it through two Flask routes, and test valid and invalid requests.

In [1]:
from pathlib import Path
import os

import ipynbname

try:
    HOMEWORK_DIR = Path(ipynbname.path()).resolve().parent
except (FileNotFoundError, IndexError):
    HOMEWORK_DIR = Path.cwd().resolve()
    candidate = HOMEWORK_DIR / "homework" / "homework13"
    if candidate.is_dir():
        HOMEWORK_DIR = candidate
os.chdir(HOMEWORK_DIR)
print("Working from:", HOMEWORK_DIR.name)

import subprocess
import sys
import time

import joblib
import requests
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

MODEL_DIR = HOMEWORK_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

Working from: homework13


## 2. Train, save, and reload

In [2]:
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)
model_path = MODEL_DIR / "model.pkl"
joblib.dump(model, model_path)
reloaded_model = joblib.load(model_path)
sample_prediction = float(reloaded_model.predict([[0.2, -0.1]])[0])
print("Saved:", model_path.relative_to(HOMEWORK_DIR))
print("Reloaded prediction:", sample_prediction)

Saved: model\model.pkl
Reloaded prediction: 10.138429495702386


## 3. Start and test the API

The server runs only for this test cell and is stopped in `finally`. The printed responses are the testing evidence.

In [3]:
server = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=HOMEWORK_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
base_url = "http://127.0.0.1:5002"
try:
    for _ in range(30):
        try:
            response = requests.get(f"{base_url}/predict/0.2/-0.1", timeout=1)
            if response.status_code == 200:
                break
        except requests.RequestException:
            time.sleep(0.2)
    else:
        raise RuntimeError("Flask server did not start")

    post_response = requests.post(
        f"{base_url}/predict", json={"features": [0.2, -0.1]}, timeout=3
    )
    get_response = requests.get(f"{base_url}/predict/0.2/-0.1", timeout=3)
    bad_response = requests.post(f"{base_url}/predict", json={"features": [0.2]}, timeout=3)

    print("POST", post_response.status_code, post_response.json())
    print("GET ", get_response.status_code, get_response.json())
    print("BAD ", bad_response.status_code, bad_response.json())
    assert post_response.status_code == get_response.status_code == 200
    assert bad_response.status_code == 400
finally:
    server.terminate()
    server.wait(timeout=5)

POST 200 {'prediction': 10.138429495702386}
GET  200 {'prediction': 10.138429495702386}
BAD  400 {'error': 'features must be a list containing exactly two values'}


## 4. Design check

`app.py` loads `model/model.pkl` once at startup. Both routes reuse that object, and invalid inputs return JSON with HTTP 400 instead of a traceback. `README.md` gives the start command and one copyable example for each route.

In [4]:
required = [
    HOMEWORK_DIR / "app.py",
    HOMEWORK_DIR / "README.md",
    HOMEWORK_DIR / "model" / "model.pkl",
]
check = {path.relative_to(HOMEWORK_DIR).as_posix(): path.exists() for path in required}
assert all(check.values())
check

{'app.py': True, 'README.md': True, 'model/model.pkl': True}